# Building Neural Networks with TensorFlow/Keras

This notebook introduces the standard TensorFlow/Keras workflow by training a multilayer perceptron (MLP) on the MNIST handwritten digit dataset.


## Learning objectives
- Load image data from `tf.keras.datasets`.
- Preprocess features and labels for dense neural networks.
- Build an MLP with the Sequential API.
- Train, evaluate, and interpret predictions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow version:", tf.__version__)


## Load the MNIST dataset
MNIST contains 70,000 grayscale images of handwritten digits. Each image is 28x28 pixels.


In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train_full.shape, X_test.shape


## Create training and validation splits
We hold out the first 5,000 training examples for validation.


In [ ]:
X_valid = X_train_full[:5000] / 255.0
X_train = X_train_full[5000:] / 255.0
X_test = X_test / 255.0

y_valid = y_train_full[:5000]
y_train = y_train_full[5000:]

X_train.shape, X_valid.shape, X_test.shape


## Inspect the class labels
The targets are digit labels from 0 through 9.


In [ ]:
class_names = [str(i) for i in range(10)]
class_names


## Visualize a few examples
Looking at the inputs helps confirm that the preprocessing pipeline is working as expected.


In [ ]:
plt.figure(figsize=(8, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i], cmap="binary")
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()


## Flatten each image for an MLP
Dense networks expect one-dimensional feature vectors, so we reshape each 28x28 image into 784 inputs.


In [ ]:
X_train_flat = X_train.reshape(-1, 28 * 28)
X_valid_flat = X_valid.reshape(-1, 28 * 28)
X_test_flat = X_test.reshape(-1, 28 * 28)

X_train_flat.shape


## Build the model
This Sequential model uses two hidden dense layers with ReLU activations and a softmax output layer.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(300, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.summary()


## Compile the model
We use sparse categorical crossentropy because the labels are integer-encoded.


In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])


## Train the network
Training history stores accuracy and loss for both training and validation splits.


In [ ]:
history = model.fit(
    X_train_flat, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_valid_flat, y_valid),
    verbose=2
)


## Plot learning curves
These curves help diagnose underfitting and overfitting.


In [ ]:
history_df = tf.keras.utils.plot_model if False else None
metrics = history.history
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(metrics["loss"], label="train")
plt.plot(metrics["val_loss"], label="valid")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(metrics["accuracy"], label="train")
plt.plot(metrics["val_accuracy"], label="valid")
plt.title("Accuracy")
plt.legend()
plt.tight_layout()


## Evaluate on the test set
Final test performance estimates how well the MLP generalizes to unseen digits.


In [ ]:
test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


## Generate predictions
`model.predict()` returns class probabilities for each image.


In [ ]:
pred_probs = model.predict(X_test_flat[:5])
pred_labels = np.argmax(pred_probs, axis=1)
print("Predicted:", pred_labels)
print("Actual:   ", y_test[:5])


## Inspect predicted probabilities for one example
The highest-probability class becomes the predicted label.


In [ ]:
sample_idx = 0
plt.imshow(X_test[sample_idx], cmap="binary")
plt.axis("off")
plt.show()

for digit, prob in enumerate(pred_probs[sample_idx]):
    print(f"{digit}: {prob:.4f}")


## Next steps
Try changing the number of hidden layers, adding dropout, or using callbacks such as `EarlyStopping` to improve generalization.
